# Explore Keele campus energy data

After the shared warm-up, use this notebook to investigate your chosen question in [TASK.md](../TASK.md). See [the setup guide](README.md) if you have not run a notebook before.

Run sections 1–4 from top to bottom. Then choose an optional example that helps with your question. Code cells have a run button; their tables and plots appear underneath. Press **Shift+Enter** to run a cell and move to the next one.

When you change a setting, rerun that cell and the cells below it. **Restart Kernel**, followed by **Run All**, gives a fresh run and avoids mixing old results with new choices.

## 1. Choose what to investigate

Change the settings below. Start with the default week, then try a different week or measurement. Keep quotation marks around dates and column names. When changing `YEAR`, change the dates too.

| Measurement | Column name |
| --- | --- |
| Campus consumption | `power-con-ave` |
| Wind generation | `power-gen-wt-ave` |
| Solar generation | `power-gen-pv-ave` |

`END` is excluded, so 1 June to 8 June selects seven days.

In [ ]:
YEAR = 2023  # Try 2024 or 2025, and change the dates to match.
START = "2023-06-01"
END = "2023-06-08"
COLUMN = "power-gen-pv-ave"  # Try "power-con-ave" or "power-gen-wt-ave".

## 2. Load and check the data

Run the next cell as it is. It finds the data folder, checks the five-minute readings and selects your dates. You can expand the code if you want to see how it works.

Check the reported timestamps and row count. The default selection should contain **2,016 rows covering 168 hours**. A file may not cover your entire requested period.

`data` contains the whole annual file; `period` contains your selected rows. These examples use the supplied timestamps without shifting them or assuming whether they label the start or end of an interval.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Works when the notebook starts in either the project folder or code folder.
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "2023-Keele-Campus-Energy-Data.csv").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("Open the extracted ERA-event folder in VS Code, then reopen this notebook.")

file = PROJECT / "data" / f"{YEAR}-Keele-Campus-Energy-Data.csv"
data = pd.read_csv(file)
data["DateTime"] = pd.to_datetime(data["DateTime"], errors="raise")
power_columns = ["power-con-ave", "power-gen-wt-ave", "power-gen-pv-ave"]
data[power_columns] = data[power_columns].apply(pd.to_numeric, errors="raise")

# Stop if missing readings or irregular times would make our sums misleading.
if data[["DateTime"] + power_columns].isna().any().any():
    raise ValueError("Missing values found. Ask a facilitator before calculating.")
gaps = data["DateTime"].diff().iloc[1:]
if not gaps.eq(pd.Timedelta(minutes=5)).all():
    raise ValueError("Timestamps are not five minutes apart. Ask a facilitator.")
if COLUMN not in power_columns:
    raise ValueError("Choose COLUMN from the three names in power_columns.")
if pd.Timestamp(START) >= pd.Timestamp(END):
    raise ValueError("START must be earlier than END.")

# Each row is one observation. Keep rows at or after START and before END.
period = data.loc[
    (data["DateTime"] >= START) & (data["DateTime"] < END)
].copy()
if period.empty:
    raise ValueError("No rows selected. Check YEAR, START and END.")

print(f"File: {file.name}")
print(f"Available timestamps: {data['DateTime'].min()} to {data['DateTime'].max()}")
print(f"Selected timestamps: {period['DateTime'].min()} to {period['DateTime'].max()}")
print(f"Selected rows: {len(period):,}; represented hours: {len(period) / 12:.2f}")
display(period.head())

## 3. See your first result

Run the cell to plot your selected measurement and find its mean and maximum. Before changing the week or column, predict how the plot might change. Then return to section 1, change one choice and rerun sections 1–3.

In [ ]:
values = period[COLUMN]
print(f"Mean: {values.mean():.2f} kW")
print(f"Maximum: {values.max():.2f} kW")
peak_row = values.idxmax()
print(f"First timestamp at that maximum: {period.loc[peak_row, 'DateTime']}")
ax = period.plot(x="DateTime", y=COLUMN, figsize=(10, 4), legend=False)
ax.set(title=f"{COLUMN}: {START} to before {END}", xlabel="Supplied timestamp", ylabel="Power (kW)")
plt.tight_layout()
plt.show()

## 4. Try a rolling average

A rolling average smooths short-term changes by averaging a moving window of readings. Change `WINDOW_HOURS` to **1**, **6** or **24**, then rerun this cell. What becomes easier to see? Which peaks disappear?

Here the average uses the current and preceding readings. The line starts only when a full window is available. This describes historical data; it is not a forecast.

In [ ]:
WINDOW_HOURS = 6  # Try 1, 6 or 24 hours. Each hour contains 12 readings.
window_rows = int(WINDOW_HOURS * 12)
if WINDOW_HOURS <= 0 or window_rows != WINDOW_HOURS * 12:
    raise ValueError("Choose a positive window in whole five-minute steps, such as 1, 6 or 24 hours.")
if window_rows > len(period):
    raise ValueError("The window is longer than your selected period. Reduce it or select more dates.")
smoothed = period[COLUMN].rolling(window_rows, min_periods=window_rows).mean()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(period["DateTime"], period[COLUMN], alpha=0.4, label="Five-minute readings")
ax.plot(period["DateTime"], smoothed, label=f"{WINDOW_HOURS}-hour rolling average")
ax.set(xlabel="Supplied timestamp", ylabel="Power (kW)", title=COLUMN)
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Choose your next step

You do not need every example below. Each uses the data loaded in sections 1–2 and can be run independently of the other optional examples.

| Your question | Start with |
| --- | --- |
| Challenge 2: compare years or months | Energy during the selected period; monthly comparison |
| Challenge 3: find excess electricity | Find excess power |
| Challenge 4: investigate zeros | Locate zeros |
| Challenge 5: explore a flexible response | Find excess power, then investigate timing and size |

Change the choices in section 1 to suit your question, rerun sections 1–2, then run your chosen example. Add your own cells using **+ Code** or **+ Markdown**.

### Energy during the selected period

Useful for Challenge 2. Every reading is five-minute average power in kW. Dividing its sum by 12,000 gives energy in MWh.

In [ ]:
energy_mwh = period[COLUMN].sum() / 12000
print(f"{COLUMN}, selected period: {energy_mwh:.3f} MWh")

What changes if you double the selected period? Can you explain why summing kW readings alone does not give kWh?

### Monthly comparison

This block uses the whole file, `data`, rather than the selected week. It reports only the readings present in each calendar month.

In [ ]:
monthly = data.set_index("DateTime")[power_columns].resample("MS").sum() / 12000
monthly.columns = ["consumption_MWh", "wind_MWh", "solar_MWh"]
monthly["renewable_to_demand_ratio"] = (
    (monthly["wind_MWh"] + monthly["solar_MWh"])
    / monthly["consumption_MWh"].replace(0, float("nan"))
)
display(monthly.round(3))
print("Readings per month:")
display(data.set_index("DateTime").resample("MS").size().rename("readings"))

monthly_plot = monthly[["consumption_MWh", "wind_MWh", "solar_MWh"]].copy()
monthly_plot.index = monthly_plot.index.strftime("%Y-%m")
ax = monthly_plot.plot.bar(figsize=(10, 4))
ax.set(xlabel="Month", ylabel="Energy (MWh)")
plt.tight_layout()
plt.show()

Which months are complete? A boundary month may contain only a few readings. Compare the same months in another year by changing `YEAR`. The ratio compares total generation with total demand; it does not say how much demand was supplied by renewables at the same time.

### Find excess power

Useful for Challenges 3 and 5. Work out the sign of generation minus demand before running this block.

In [ ]:
period["generation_kW"] = period["power-gen-wt-ave"] + period["power-gen-pv-ave"]
period["balance_kW"] = period["generation_kW"] - period["power-con-ave"]
period["excess_kW"] = period["balance_kW"].clip(lower=0)
excess_rows = period["excess_kW"] > 0
print(f"Hours with excess: {excess_rows.sum() / 12:.2f}")
print(f"Excess energy: {period['excess_kW'].sum() / 12000:.3f} MWh")
print(f"Largest five-minute average excess: {period['excess_kW'].max():.2f} kW")

ax = period.plot(x="DateTime", y=["power-con-ave", "generation_kW", "excess_kW"], figsize=(10, 4))
ax.set(xlabel="Supplied timestamp", ylabel="Power (kW)")
plt.tight_layout()
plt.show()

`.clip(lower=0)` replaces negative balances with zero. Why would adding negative balances give a different answer? The hours here may be scattered across many events; they are not the duration of one continuous event. Inspect demand and generation before trusting an apparent surplus. This calculation alone does not size a battery or electrolyser.

### Locate zeros

Useful for Challenge 4. Choose the measurement at the top of the starter.

In [ ]:
zero_rows = period[COLUMN] == 0
print(f"Zero readings: {zero_rows.sum()} out of {len(period)}")
display(period.loc[zero_rows, ["DateTime", COLUMN]].head(20))

Only the first 20 matching rows are displayed. What would you plot or compare to investigate the zeros? This block counts them; it does not identify failures, detect every quality problem or replace readings.

### Save selected data for VS Plotter

Optional: change `SAVE_CSV` to `True` and run the cell to save your selected rows, including any columns you calculated. Open `code/selected-period.csv` with VS Plotter. Select `DateTime` as X and your measurements as Y.

This leaves the supplied data unchanged. Exporting again replaces `selected-period.csv`; rename that output first if you want to keep it.

In [ ]:
SAVE_CSV = False  # Change to True only when you want to write the CSV.
if SAVE_CSV:
    output = PROJECT / "code" / "selected-period.csv"
    period.to_csv(output, index=False)
    print(f"Saved {output.name} in the code folder.")

## Your investigation

Double-click this text cell and replace the prompts with your notes. Press Shift+Enter to display the text again.

- **Our question:**
- **What we changed or compared:**
- **Our finding and the evidence:**
- **Assumptions, limitations and what we would check next:**

Use the code cell below to extend an example or try your own calculation. Save the notebook with **Ctrl+S** to keep your code, notes and displayed results.

For your group presentation, choose one clearly labelled plot and one finding. If you share your work or continue it after the event, acknowledge the data and paper as explained in [ACKNOWLEDGEMENTS.md](../ACKNOWLEDGEMENTS.md).

In [ ]:
# Add your own analysis here.